# Controllable Image Generation for Inverse Problems — Colab demo

This notebook runs the full benchmark end to end on a Colab GPU: six inverse problems,
two generative models (JiT and pMF), and every reconstruction strategy in the registry,
all on the **same** problem instances with the **same** generative noise.

**Runtime.** `Runtime → Change runtime type → GPU`.

| GPU | Status |
|---|---|
| **A100** | best; the default config finishes in a few minutes |
| **L4** | comfortable; recommended for routine use |
| **T4** | works, but substantially slower — reduce `num_images` or enable fewer tasks |
| CPU | structural checks only; not usable for real runs |

The whole notebook is: clone → setup → **restart once** → run. Nothing else.


## 0. Confirm the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# compute_cap >= 8.0 means BF16 is available, which JiT prefers.
# On older GPUs the code falls back to FP32 (correct, just slower) and never FP16,
# which is deliberate: FP16 destabilises pixel-space generation.


## 1. Clone the repository

Replace the URL with your fork if you have one.


In [ ]:
!git clone https://github.com/Abdelaal495/controllable-image-generation.git
%cd controllable-image-generation


## 2. Install dependencies

`setup_colab.sh` detects the accelerator, installs the JAX and PyTorch stacks in an order
that works on Colab, repairs Pillow, and clones the four model repositories at pinned
revisions. It is idempotent — re-running it is cheap.


In [ ]:
!bash setup_colab.sh


## 3. ⚠️ RESTART THE RUNTIME NOW

**`Runtime → Restart session`, then continue from the next cell.**

Installing JAX replaces shared libraries that the running Python process has already
imported. Without a restart you will get confusing `jaxlib` errors. This is needed
**once**; re-running the setup afterwards is a no-op because it leaves a marker file.


## 4. After the restart


In [ ]:
%cd /content/YOUR-REPO
!pwd


## 5. Hugging Face token

ImageNet-1k is a **gated** dataset. Before the first run:

1. accept the licence at <https://huggingface.co/datasets/ILSVRC/imagenet-1k>
2. create a token at <https://huggingface.co/settings/tokens>
3. add it as a Colab secret named `HF_TOKEN` (🔑 icon in the left sidebar) and enable
   notebook access

The code reads Colab secrets, then the environment, then a `.env` file — in that order.
No token is needed if you switch to `data.source: local_folder`.


In [ ]:
from google.colab import userdata
try:
    token = userdata.get('HF_TOKEN')
    print('HF_TOKEN found:', bool(token), '| length', len(token) if token else 0)
except Exception as exc:
    print('No Colab secret named HF_TOKEN:', exc)
    print('Add one via the key icon in the sidebar, or use data.source: local_folder.')


## 6. Dry run — validate the plan without loading a model

This resolves the configuration, expands every sweep, prints the atomic jobs and the
warnings, and exits. Always worth a look before committing GPU time: it catches typos,
invalid combinations and accidentally enormous sweeps in seconds.


In [ ]:
%run run.py --config configs/experiments.yaml --dry-run


## 7. Run

`%run` (not `!python`) gives notebook-like behaviour: progress, summary tables and
matplotlib figures appear inline.

Roughly 6–15 minutes on an A100 or L4 with the default config; longer on a T4.


In [ ]:
%run run.py --config configs/experiments.yaml


## 8. Results

Everything lands in `outputs/<run_id>/`. Each atomic job is written the moment it
finishes, so a disconnect never costs you completed work.


In [ ]:
import glob, os
run_dir = sorted(glob.glob('outputs/run_*'))[-1]
print('run directory:', run_dir)
for name in sorted(os.listdir(run_dir)):
    print('  ', name)


In [ ]:
import pandas as pd
df = pd.read_csv(f'{run_dir}/results.csv')
ok = df[df.status == 'ok']
cols = ['task', 'model', 'method', 't0', 'steps', 'num_mpc_steps', 'K', 'lam',
        'psnr', 'ssim', 'lpips', 'measurement_rmse', 'runtime_per_image']
ok[cols].sort_values(['task', 'model', 'method'])


### Figures

* `paired_deltas.png` — each MPC job minus its **step-matched** SDEdit baseline
* `configurations_<model>.png` — every configuration separately, with a dashed line for
  the degraded observation (i.e. doing nothing at all)
* `<task>_<model>_page_01.png` — the actual reconstructions
* `problem_instances.png` — ground truth / degraded / guide, per task


In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob(f'{run_dir}/figures/*.png')):
    print(path)
    display(Image(filename=path))


## 9. Where to go next

Everything below is a `configs/experiments.yaml` edit — no code changes.

**Corruption-strength sweep.** The default sits at `t0: 0.8`, which destroys 80% of the
signal; most strategies score below the degraded input there. Finding where each one
crosses that line is the more informative experiment:

```yaml
sdedit:      {t0: [0.3, 0.5, 0.6, 0.8], steps: 4, solver: heun}
mpc_rhc:     {t0: [0.3, 0.5, 0.6, 0.8], num_mpc_steps: 4, K: 1}
mpc_delta_t: {t0: [0.3, 0.5, 0.6, 0.8], num_mpc_steps: 4}
```

**Planning-horizon sweep.** `num_mpc_steps: [2, 4, 8, 16]` tests whether the standard-flow
value approximation is the limiting factor relative to the MeanFlow one.

**Regularisation sweep.** `lam: [0.5, 1, 5, 15]` — the built-in defaults come from the
MPC-Flow paper's Table E2, which was tuned for a different model and resolution.

Always check the size first:


In [ ]:
%run run.py --config configs/experiments.yaml --dry-run


---

**Colab tips**

* Mount Drive and set `runtime.output_root` to a Drive path to keep results across sessions.
* `--num-images 1 --experiments denoising` is a fast smoke test.
* `--models jit` restricts to PyTorch only (skips JAX entirely).
* Re-running with `--run-id <existing>` reuses finished jobs instead of recomputing them.

For clusters (Alliance / Compute Canada), see `docs/quickstart_cluster.md`.
